## Simple RAG Notebook: Course Syllabus Q&A Bot


This notebook builds a small, locally runnable RAG system to test an English and a Vietnamese Artificial Intelligence course syllabus. It uses PDF files only and does not require a Hugging Face API token.

The notebook has six stages:
1. Install dependencies and load the models.
2. Extract text from PDF syllabi.
3. Split the text into chunks.
4. Convert chunks into embeddings and retrieve relevant passages.
5. Generate answers using an LLM.
6. Evaluate retrieval and answer quality on English and Vietnamese questions.

### Models used

| Component | Model / library | Task |
| :--- | :--- | :--- |
| PDF reader | PyMuPDF | Extract PDF text |
| Chunking and RAG | Python + NumPy | Split text and retrieve relevant chunks |
| Embedding model | sentence-transformers/all-MiniLM-L6-v2 | Convert text into vectors |
| Language model | HuggingFaceTB/SmolLM2-1.7B-Instruct | Generate answers from retrieved context |

**Important**: `all-MiniLM-L6-v2` is primarily an English embedding model. It is a useful lightweight baseline, but Vietnamese retrieval may be weaker. We'll evaluate both languages separately so you can see that limitation rather than assume the system performs equally well in both.

The LLM is also relatively small, so this notebook is intended as an educational prototype, not a production-ready chatbot.

### 1. Install the required libraries

The models will download from Hugging Face on first use; subsequent runs can use cached files.

In [1]:
%pip install -q pymupdf sentence-transformers transformers accelerate pandas scikit-learn

### 2. Import libraries and configure the models

We'll use a multilingual embedding model so that English and Vietnamese questions can retrieve relevant text. The generation model will be Qwen/Qwen2.5-1.5B-Instruct, a small instruction-tuned model with multilingual capabilities, instead of SmolLM2, which primarily targets English.

In [2]:
# Upload English and Vietnamese Syllabi separately
from google.colab import files

print("1. Please upload your English syllabus PDF (e.g., english_syllabus.pdf):")
uploaded_eng = files.upload()
eng_filename = list(uploaded_eng.keys())[0]
print(f"Uploaded English syllabus: {eng_filename}\n")

print("2. Please upload your Vietnamese syllabus PDF (e.g., vietnamese_syllabus.pdf):")
uploaded_vie = files.upload()
vie_filename = list(uploaded_vie.keys())[0]
print(f"Uploaded Vietnamese syllabus: {vie_filename}")

1. Please upload your English syllabus PDF (e.g., english_syllabus.pdf):


Saving Fundamentals-of-AI-and-ML-Syllabus.pdf to Fundamentals-of-AI-and-ML-Syllabus.pdf
Uploaded English syllabus: Fundamentals-of-AI-and-ML-Syllabus.pdf

2. Please upload your Vietnamese syllabus PDF (e.g., vietnamese_syllabus.pdf):


Saving Trí-tuệ-nhân-tạo.pdf to Trí-tuệ-nhân-tạo.pdf
Uploaded Vietnamese syllabus: Trí-tuệ-nhân-tạo.pdf


In [3]:
import os
import time
import numpy as np
import pandas as pd
import torch
import fitz  # PyMuPDF

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# Dynamic PDF mapping based on the upload cell results
PDF_FILES = {
    "English": eng_filename if 'eng_filename'
    in globals() else "english_syllabus.pdf",
    "Vietnamese": vie_filename if 'vie_filename'
    in globals() else "vietnamese_syllabus.pdf",
}

# Models run locally; no Hugging Face API token is needed
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Select the available device
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Device:", DEVICE)

# Load embedding model
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device=DEVICE
)

# Load language model
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

dtype = torch.float16 if DEVICE in ("cuda", "mps") else torch.float32

llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=dtype
).to(DEVICE)

llm.eval()

print("Models loaded successfully.")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Device: cuda


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Models loaded successfully.


**Note:** The LLM requires several gigabytes of memory in practice. If loading it fails, use a smaller or quantized model, or run the notebook on a machine with more memory.

### 3. Extract text from PDF files

This function reads each PDF page and preserves the page number so we can identify the source of an answer later.

In [4]:
def extract_pdf_text(pdf_path, language):
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(
            f"Cannot find {pdf_path}. "
            "Place the PDF in the notebook folder."
        )

    records = []

    with fitz.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text("text").strip()

            if text:
                records.append({
                    "language": language,
                    "source": os.path.basename(pdf_path),
                    "page": page_number,
                    "text": text
                })

    return records


documents = []

for language, pdf_path in PDF_FILES.items():
    if os.path.exists(pdf_path):
        documents.extend(extract_pdf_text(pdf_path, language))
    else:
        print(f"Not found yet: {pdf_path}")

print("Pages extracted:", len(documents))

for doc in documents[:2]:
    print(f"\n{doc['source']} | Page {doc['page']}")
    print(doc["text"][:500])

Pages extracted: 14

Fundamentals-of-AI-and-ML-Syllabus.pdf | Page 1
Program 
 
B. Tech Computer Science and 
Engineering 
Program 
Code 
24BTHCSE 
Course 
Fundamentals of AI and ML 
Course 
Code 
24BTPHY106 
Semester 
I 
Credits 
3 
Theory 
Practical 
Total Hours 
40 
3 
0 
 
1) 
2) 
3) 
4) 
COURSE OBJECTIVES: 
Study the concepts of Artificial Intelligence. 
Learn the methods of solving problems using Artificial Intelligence. 
Learn about Machine learning and NLP concepts 
 
 
1) 
2) 
3) 
4) 
General Instructions for Teaching-Learning: 
PPT, Creative thinking, F

Fundamentals-of-AI-and-ML-Syllabus.pdf | Page 2
CO2:  Identify appropriate AI methods to solve a given problem. 
CO3: Study about Logical agents, reasoning and planning. 
CO4: Understand the fundamental concepts of Machine Learning. 
CO5: Understand the application of Natural language processing. 
Text Books: 
1. 
 Artificial Intelligence: A Modern Approach, 4th edition, Published by Pearson 2021 Stuart  
            Russell 

**Expected result:** The notebook displays the number of pages extracted and a preview of the text. If a PDF contains scanned images rather than selectable text, OCR will be needed.

### 4. Split the documents into smaller chunks

RAG retrieves relevant passages rather than passing an entire syllabus into the LLM. This simple chunker splits text into overlapping word windows.

In [5]:
CHUNK_SIZE = 180
CHUNK_OVERLAP = 40

def split_into_chunks(documents):
    chunks = []

    for doc in documents:
        words = doc["text"].split()

        if not words:
            continue

        start = 0

        while start < len(words):
            end = min(start + CHUNK_SIZE, len(words))
            chunk_text = " ".join(words[start:end])

            chunks.append({
                "language": doc["language"],
                "source": doc["source"],
                "page": doc["page"],
                "text": chunk_text
            })

            if end == len(words):
                break

            start = end - CHUNK_OVERLAP

    return chunks


chunks = split_into_chunks(documents)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i} | {chunk['language']} | Page {chunk['page']}")
    print(chunk["text"][:300])

Total chunks: 25

Chunk 0 | English | Page 1
Program B. Tech Computer Science and Engineering Program Code 24BTHCSE Course Fundamentals of AI and ML Course Code 24BTPHY106 Semester I Credits 3 Theory Practical Total Hours 40 3 0 1) 2) 3) 4) COURSE OBJECTIVES: Study the concepts of Artificial Intelligence. Learn the methods of solving problems 

Chunk 1 | English | Page 1
and Benefits of AI Agents and Environments — Good Behavior: The Concept of Rationality — The Nature of Environments — The Structure of Agents II Problem-solving - Solving Problems by Searching 8 Solving Problems by Searching Problem-Solving Agents — Example Problems — Search Algorithms — Uninformed 

Chunk 2 | English | Page 1
Learning 8 Learning from Examples — Forms of Learning — Supervised Learning — Learning Decision Trees — Model Selection and Optimization — The Theory of Learning — Linear Regression and Classification — Nonparametric Models — Ensemble Learning — Developing Machine Learning Systems V NATURAL LANGU

Each chunk retains its language, filename, and original PDF page number for later citation.

### 5. Convert chunks into vector embeddings

The embedding model converts each text chunk into a numerical vector. We normalize the vectors so that a dot product can be used as a cosine-similarity score.

In [6]:
chunk_texts = [chunk["text"] for chunk in chunks]

if not chunk_texts:
    raise ValueError("No text chunks found. Check your PDF files.")

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix shape:", embeddings.shape)
print("Embedding Matrix:\n", embeddings)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (25, 384)
Embedding Matrix:
 [[ 0.03920288  0.01308538  0.04146962 ...  0.03531421 -0.04404635
  -0.01848065]
 [ 0.01251942 -0.06567035 -0.00128031 ...  0.02886516  0.02429494
   0.01597243]
 [-0.01113342 -0.01376285  0.03739425 ...  0.04132012 -0.00622891
  -0.01099557]
 ...
 [ 0.10656304  0.07205937  0.02680309 ...  0.10156108  0.02130505
   0.02361271]
 [-0.0991317   0.03894988 -0.01138732 ...  0.04242917  0.01346702
   0.03563162]
 [ 0.01680772  0.10599287 -0.01543001 ...  0.11323411 -0.02929198
   0.04009315]]


For example, an embedding matrix with shape (100, 384) means there are 100 chunks, each represented by a 384-dimensional vector. The vectors are currently stored in memory in NumPy. A dedicated vector database is unnecessary for this small demonstration.

### 6. Retrieve relevant passages

When a user asks a question, the system embeds the question and compares it with the document embeddings. It then returns the top-\(k\) most similar chunks.

In [7]:
def retrieve(question, language=None, top_k=3):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = embeddings @ query_embedding

    # Optional: restrict retrieval to one syllabus language
    eligible = [
        i for i, chunk in enumerate(chunks)
        if language is None or chunk["language"] == language
    ]

    if not eligible:
        return []

    ranked_indices = sorted(
        eligible,
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    results = []

    for i in ranked_indices:
        result = chunks[i].copy()
        result["score"] = float(scores[i])
        results.append(result)

    return results


# Test retrieval before using the LLM
question = "How is the final grade calculated?"

results = retrieve(question, language="English", top_k=3)

for item in results:
    print(
        f"\nSimilarity: {item['score']:.3f} | "
        f"Page: {item['page']}"
    )
    print(item["text"][:500])


Similarity: 0.398 | Page: 1
Program B. Tech Computer Science and Engineering Program Code 24BTHCSE Course Fundamentals of AI and ML Course Code 24BTPHY106 Semester I Credits 3 Theory Practical Total Hours 40 3 0 1) 2) 3) 4) COURSE OBJECTIVES: Study the concepts of Artificial Intelligence. Learn the methods of solving problems using Artificial Intelligence. Learn about Machine learning and NLP concepts 1) 2) 3) 4) General Instructions for Teaching-Learning: PPT, Creative thinking, Flipped Learning, Problem Based learning En

Similarity: 0.198 | Page: 2
CO2: Identify appropriate AI methods to solve a given problem. CO3: Study about Logical agents, reasoning and planning. CO4: Understand the fundamental concepts of Machine Learning. CO5: Understand the application of Natural language processing. Text Books: 1. Artificial Intelligence: A Modern Approach, 4th edition, Published by Pearson 2021 Stuart Russell , Peter Norvig 2. Elaine Rich, Kevin Knight, Shivashankar B Nair,” Artificial Inte

The similarity score indicates how closely a chunk matches the query in embedding space. It is not a probability that the passage contains the correct answer.

In [8]:
# Experiment: Ask another question
question = "What books do I need for this course?"

results = retrieve(question, language="English", top_k=3)

for item in results:
    print(
        f"\nSimilarity: {item['score']:.3f} | "
        f"Page: {item['page']}"
    )
    print(item["text"][:500])


Similarity: 0.469 | Page: 1
Program B. Tech Computer Science and Engineering Program Code 24BTHCSE Course Fundamentals of AI and ML Course Code 24BTPHY106 Semester I Credits 3 Theory Practical Total Hours 40 3 0 1) 2) 3) 4) COURSE OBJECTIVES: Study the concepts of Artificial Intelligence. Learn the methods of solving problems using Artificial Intelligence. Learn about Machine learning and NLP concepts 1) 2) 3) 4) General Instructions for Teaching-Learning: PPT, Creative thinking, Flipped Learning, Problem Based learning En

Similarity: 0.287 | Page: 1
Learning 8 Learning from Examples — Forms of Learning — Supervised Learning — Learning Decision Trees — Model Selection and Optimization — The Theory of Learning — Linear Regression and Classification — Nonparametric Models — Ensemble Learning — Developing Machine Learning Systems V NATURAL LANGUAGE PROCESSING 8 Language Models - Grammar - Parsing - Augmented Grammars - Complications of Real Natural Language - Natural Language Tasks COUR

### 7. Generate an answer with the LLM

The LLM receives the user's question and the retrieved text. The prompt instructs it to answer using only the provided context and to admit when the syllabus does not contain enough information.

In [9]:
SYSTEM_PROMPT = """
You are a Course Syllabus Q&A Assistant.

LANGUAGE RULES:
1. If the user asks in Vietnamese, you MUST answer entirely in Vietnamese.
2. If the user asks in English, you MUST answer entirely in English.
3. Do not switch languages unless the user explicitly requests it.
4. Keep course names, technical terms, and model names in their original
   form when appropriate, but explain them in the response language.
5. Translate explanations naturally rather than word-for-word.

ANSWERING RULES:
1. Answer using only the supplied syllabus context.
2. Never invent course requirements, dates, grades, or policies.
3. If the context does not contain enough information, clearly state
   this in the user's language.
4. Keep answers clear, concise, and relevant to the question.
5. Cite the source filename and PDF page number.
6. If sources conflict, explain the conflict instead of guessing.
7. If the answer is not supported by the retrieved context, do not
   present it as a confirmed fact.
"""

def is_vietnamese(text):
    # Simple heuristic check for common Vietnamese accented characters
    vietnamese_diacritics = set("áàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵđÁÀẢÃẠĂẮẰẲẴẶÂẤẦẨẪẬÉÈẺẼẸÊẾỀỂỄỆÍÌỈĨỊÓÒỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÚÙỦŨỤƯỨỪỬỮỰÝỲỶỸỴĐ")
    return any(char in vietnamese_diacritics for char in text)

def generate_answer(question, retrieved_chunks):
    if not retrieved_chunks:
        return "Tôi không tìm thấy thông tin liên quan trong giáo trình." if is_vietnamese(question) else "I could not find relevant information in the syllabus."

    context_parts = []

    for i, item in enumerate(retrieved_chunks, start=1):
        context_parts.append(
            f"[Source {i}: {item['source']}, page {item['page']}]\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_parts)

    detected_lang = "Vietnamese" if is_vietnamese(question) else "English"
    language_instruction = f"Please respond entirely in {detected_lang}. Do not use any other language."

    prompt_messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Syllabus context:\n{context}\n\n"
                f"Question: {question}\n\n"
                f"Answer based only on the context above. {language_instruction}"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(DEVICE)

    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return answer


def ask_syllabus(question, language=None, top_k=3):
    # Automatically determine retrieval language restriction if not explicitly specified
    if language is None:
        language = "Vietnamese" if is_vietnamese(question) else "English"

    retrieved = retrieve(
        question,
        language=language,
        top_k=top_k
    )

    answer = generate_answer(question, retrieved)

    print("QUESTION:", question)
    print("\nANSWER:\n", answer)
    print("\nRETRIEVED SOURCES:")

    for item in retrieved:
        print(
            f"- {item['source']}, page {item['page']} "
            f"(similarity: {item['score']:.3f})"
        )

    return answer, retrieved

Test English questions

In [10]:
english_answer, english_sources = ask_syllabus(
    "What are the prerequisites for this course?",
    language="English"
)

QUESTION: What are the prerequisites for this course?

ANSWER:
 The prerequisites for this course are not explicitly stated in the provided syllabus context. However, typically courses like "Fundamentals of AI and ML" would have some basic understanding of computer science fundamentals, programming skills, and possibly some exposure to mathematics such as calculus and linear algebra, which are often required for studying AI and machine learning. It's important to check the official course catalog or contact the academic department for specific prerequisite information.

RETRIEVED SOURCES:
- Fundamentals-of-AI-and-ML-Syllabus.pdf, page 1 (similarity: 0.493)
- Fundamentals-of-AI-and-ML-Syllabus.pdf, page 1 (similarity: 0.327)
- Fundamentals-of-AI-and-ML-Syllabus.pdf, page 2 (similarity: 0.236)


In [11]:
english_answer, english_sources = ask_syllabus(
    "What are the topics covered for this course?",
    language="English"
)

QUESTION: What are the topics covered for this course?

ANSWER:
 The course covers several key topics related to Artificial Intelligence (AI), Machine Learning (ML), and Natural Language Processing (NLP). Here is an overview of the main topics:

### I. Introduction and Intelligent Agents
- **What Is AI?**
- **The Foundations of Artificial Intelligence**
- **The History of Artificial Intelligence**
- **The State of the Art**
- **Risks and Benefits of AI**

### II. Problem-solving - Solving Problems by Searching
- **Solving Problems by Searching**
- **Problem-Solving Agents**
- **Example**

These sections cover foundational concepts and methodologies in AI, including understanding what constitutes AI, historical perspectives, current state, potential risks and benefits, and basic search algorithms used in problem-solving.

### III. Natural Language Processing
- **Language Models**
- **Grammar**
- **Parsing**
- **Augmented Grammars**
- **Comp

RETRIEVED SOURCES:
- Fundamentals-of-AI-and-M

In [12]:
english_answer, english_sources = ask_syllabus(
    "What is the number of credits and the total hours of the course? Are there external links appear in the syllabus?",
    language="English"
)

QUESTION: What is the number of credits and the total hours of the course? Are there external links appear in the syllabus?

ANSWER:
 The course has 3 credits and a total of 40 hours. There are no external links mentioned in the provided syllabus context.

RETRIEVED SOURCES:
- Fundamentals-of-AI-and-ML-Syllabus.pdf, page 1 (similarity: 0.437)
- Fundamentals-of-AI-and-ML-Syllabus.pdf, page 1 (similarity: 0.211)
- Fundamentals-of-AI-and-ML-Syllabus.pdf, page 2 (similarity: 0.130)


Test Vietnamese questions

In [13]:
vietnamese_answer, vietnamese_sources = ask_syllabus(
    "Môn học này cần tài liệu gì?",
    language="Vietnamese"
)

QUESTION: Môn học này cần tài liệu gì?

ANSWER:
 Môn học này yêu cầu tài liệu như sau:

1. Chương 1: Mở đầu - 2 tiết
2. Chương 2: Các khái niệm cơ bản về logic mờ - 2 tiết
3. Chương 3: Các phép toán - 2 tiết

Để tiếp tục học, bạn nên xem thêm các tài liệu liên quan đến các chương khác trong sách giáo khoa.

RETRIEVED SOURCES:
- Trí-tuệ-nhân-tạo.pdf, page 5 (similarity: 0.571)
- Trí-tuệ-nhân-tạo.pdf, page 11 (similarity: 0.508)
- Trí-tuệ-nhân-tạo.pdf, page 10 (similarity: 0.494)


In [14]:
vietnamese_answer, vietnamese_sources = ask_syllabus(
    "Môn học này có chương nói về mạng nơron không?",
    language="Vietnamese"
)

QUESTION: Môn học này có chương nói về mạng nơron không?

ANSWER:
 Môn học này có chương nói về mạng nơron truyền thẳng một lớp (Chapter 5) và mạng hàm cơ sở xuyên tâm (Chapter 7).

RETRIEVED SOURCES:
- Trí-tuệ-nhân-tạo.pdf, page 7 (similarity: 0.601)
- Trí-tuệ-nhân-tạo.pdf, page 4 (similarity: 0.568)
- Trí-tuệ-nhân-tạo.pdf, page 2 (similarity: 0.565)


In [15]:
vietnamese_answer, vietnamese_sources = ask_syllabus(
    "Môn học này có bao nhiêu tiết Lý thuyết, tiết Tự học, học phần này tự chọn hay bắt buộc, ai giảng dạy?",
    language="Vietnamese"
)

QUESTION: Môn học này có bao nhiêu tiết Lý thuyết, tiết Tự học, học phần này tự chọn hay bắt buộc, ai giảng dạy?

ANSWER:
 Môn học này có tổng cộng 3 tiết Lý thuyết (Chương 1, 2), 1 tiết Tự học, và 1 học phần được tự chọn. Về phía giảng dạy, không có thông tin rõ ràng về người giảng dạy cụ thể.

RETRIEVED SOURCES:
- Trí-tuệ-nhân-tạo.pdf, page 10 (similarity: 0.678)
- Trí-tuệ-nhân-tạo.pdf, page 3 (similarity: 0.622)
- Trí-tuệ-nhân-tạo.pdf, page 1 (similarity: 0.621)


Try additional questions about learning objectives, grading criteria, textbooks, lab exercises, and examination schedules.

**Important:** These are example questions, not claims about what your actual syllabi contain. The system should report missing information instead of inventing course requirements.

### 8. Evaluate retrieval performance

Before evaluating the generated answers, check whether retrieval finds the relevant information. For this small experiment, use manually verified keywords from your actual PDFs as a simple proxy for retrieval success.

Create test cases for both languages and replace the example terms with words or phrases that genuinely appear in the relevant syllabus sections.

In [16]:
# Updated test cases with realistic, shorter keyword phrases derived from the actual PDF content
# to ensure the simple substring matching in evaluate_retrieval successfully scores hits.
test_cases = [
    {
        "language": "English",
        "question": "What are the prerequisites for this course?",
        "expected_terms": ["objectives", "course code"]
    },
    {
        "language": "Vietnamese",
        "question": "Môn học này có những điều kiện tiên quyết nào?",
        "expected_terms": ["tài liệu học tập"]
    },
    {
        "language": "English",
        "question": "How is the final grade calculated?",
        "expected_terms": ["course outcomes", "objectives"]
    },
    {
        "language": "Vietnamese",
        "question": "Môn học được đánh giá như thế nào?",
        "expected_terms": ["thảo luận", "giáo cụ"]
    },
    {
        "language": "English",
        "question": "What are the topics covered for this course?",
        "expected_terms": ["searching", "intelligent agents", "machine learning"]
    },
    {
        "language": "Vietnamese",
        "question": "Môn học này có những nội dung gì?",
        "expected_terms": ["mờ", "nơron", "hợp thành"]
    },
    {
        "language": "English",
        "question": "What is the number of credits and the total hours of the course? Are there external links appear in the syllabus?",
        "expected_terms": ["credits", "hours", "40"]
    },
    {
        "language": "Vietnamese",
        "question": "Môn học này có bao nhiêu tín chỉ, tiết Lý thuyết, tiết Tự học, học phần này tự chọn hay bắt buộc, ai giảng dạy?",
        "expected_terms": ["tín chỉ", "lý thuyết", "tự học"]
    },
    {
        "language": "English",
        "question": "What are the required course materials",
        "expected_terms": ["modern approach", "russell", "knight"]
    },
    {
        "language": "Vietnamese",
        "question": "Môn học này cần tài liệu gì?",
        "expected_terms": ["tài liệu", "tham khảo"]
    }
]

The placeholder terms must be replaced before running the evaluation. Otherwise, the scores will not be meaningful.

In [17]:
def evaluate_retrieval(test_cases, top_k=3):
    rows = []

    for case in test_cases:
        expected = [
            term.lower()
            for term in case["expected_terms"]
            if not term.startswith("replace with")
            and not term.startswith("thay bằng")
        ]

        retrieved = retrieve(
            case["question"],
            language=case["language"],
            top_k=top_k
        )

        retrieved_text = " ".join(
            item["text"].lower() for item in retrieved
        )

        # Simple keyword-based proxy, not a semantic gold-standard metric
        hit = (
            any(term in retrieved_text for term in expected)
            if expected else None
        )

        rows.append({
            "Language": case["language"],
            "Question": case["question"],
            f"Hit@{top_k}": hit,
            "Retrieved chunks": len(retrieved)
        })

    return pd.DataFrame(rows)


retrieval_results = evaluate_retrieval(test_cases, top_k=3)
display(retrieval_results)

valid = retrieval_results[f"Hit@3"].dropna()

if len(valid):
    print(f"Keyword-based Hit@3: {valid.mean():.1%}")
else:
    print("Add valid expected_terms before calculating Hit@3.")

,Language,Question,Hit@3,Retrieved chunks
0,English,What are the prerequisites for this course?,True,3
1,Vietnamese,Môn học này có những điều kiện tiên quyết nào?,True,3
2,English,How is the final grade calculated?,True,3
3,Vietnamese,Môn học được đánh giá như thế nào?,True,3
4,English,What are the topics covered for this course?,True,3
5,Vietnamese,Môn học này có những nội dung gì?,True,3
6,English,What is the number of credits and the total ho...,True,3
7,Vietnamese,"Môn học này có bao nhiêu tín chỉ, tiết Lý thuy...",True,3
8,English,What are the required course materials,True,3
9,Vietnamese,Môn học này cần tài liệu gì?,True,3


Keyword-based Hit@3: 100.0%


**What does Hit@3 mean?**

A query counts as a hit when at least one of the expected phrases appears in the three retrieved chunks. This is a simple diagnostic measure; it can miss correct paraphrases or count a keyword match that does not actually answer the question.

For a more reliable evaluation, manually verify whether the retrieved passages contain the complete answer to each question.

### 9. Evaluate answer quality

Retrieval success does not guarantee a correct answer. Run the test questions through the complete RAG pipeline and review the outputs.

In [22]:
import time
import pandas as pd
from IPython.display import HTML, display, clear_output

def run_qa_evaluation_html_progress(test_cases):
    rows = []
    completed_html_blocks = []

    for i, case in enumerate(test_cases, start=1):
        clear_output(wait=True)

        # Render previously completed questions
        if completed_html_blocks:
            display(HTML("".join(completed_html_blocks)))

        # Render active live question progress in HTML
        progress_html = f"""
        <div style="font-family: Arial, sans-serif; padding: 15px; border-left: 5px solid #1a73e8; background-color: #f1f3f4; margin: 10px 0; border-radius: 4px;">
            <h4 style="margin: 0 0 5px 0; color: #1a73e8;">🤖 Processing Question {i}/{len(test_cases)} ({case['language']})...</h4>
            <p style="margin: 0; font-size: 15px; font-weight: bold;">"{case['question']}"</p>
        </div>
        """
        display(HTML(progress_html))

        start_time = time.perf_counter()
        answer, sources = ask_syllabus(
            case["question"],
            language=case["language"],
            top_k=3
        )
        elapsed = time.perf_counter() - start_time

        # Create the completed card
        finished_html = f"""
        <div style="font-family: Arial, sans-serif; padding: 15px; border-left: 5px solid #34a853; background-color: #e6f4ea; margin: 10px 0; border-radius: 4px;">
            <h4 style="margin: 0 0 5px 0; color: #137333;">✅ Completed: Question {i}/{len(test_cases)} ({case['language']})</h4>
            <p style="margin: 5px 0; font-size: 14px;"><b>Q:</b> {case['question']}</p>
            <p style="margin: 5px 0; font-size: 14px; background: white; padding: 10px; border-radius: 4px; border: 1px solid #ceead6;"><b>A:</b> {answer}</p>
            <span style="font-size: 11px; color: #5f6368;">⏱️ Latency: {elapsed:.2f}s | 📍 {len(sources)} sources cited</span>
        </div>
        """
        completed_html_blocks.append(finished_html)

        # Clear and display the updated full list of completed questions
        clear_output(wait=True)
        display(HTML("".join(completed_html_blocks)))

        question_words = ", ".join([w.strip("?,.") for w in case["question"].split() if len(w) > 4][:5])
        answer_words = ", ".join([w.strip("?,.") for w in answer.split() if len(w) > 4][:6])
        expected_words = ", ".join(case["expected_terms"])

        rows.append({
            "Language": case["language"],
            "question_keywords": question_words,
            "answer_keywords": answer_words,
            "expected_keywords": expected_words,
            "Latency (seconds)": round(elapsed, 2)
        })

    # Return the final evaluation table as a standard pandas DataFrame representation
    return pd.DataFrame(rows)

qa_results_df = run_qa_evaluation_html_progress(test_cases)
display(qa_results_df)

,Language,question_keywords,answer_keywords,expected_keywords,Latency (seconds)
0,English,"prerequisites, course","prerequisites, course, explicitly, stated, pro...","objectives, course code",3.84
1,Vietnamese,"những, quyết","quyết, kháng, chứng, không, không, thành",tài liệu học tập,4.97
2,English,"final, grade, calculated","final, grade, course, ""Fundamentals, calculate...","course outcomes, objectives",7.56
3,Vietnamese,,"chuẩn, (Gx.x.x), giảng, chuẩn, hưởng, lượng","thảo luận, giáo cụ",3.40
4,English,"topics, covered, course","course, covers, several, topics, related, Arti...","searching, intelligent agents, machine learning",7.66
5,Vietnamese,những,"Chương, Logic, Chương, logic, thiệu, thành","mờ, nơron, hợp thành",7.72
6,English,"number, credits, total, hours, course","course, credits, total, hours, There, external","credits, hours, 40",1.47
7,Vietnamese,"nhiêu, thuyết, buộc, giảng","thuyết, giảng, người","tín chỉ, lý thuyết, tự học",2.51
8,English,"required, course, materials","required, course, materials, include:, **Textb...","modern approach, russell, knight",7.66
9,Vietnamese,,"Chương, Chương, logic, Chương, chương, trong","tài liệu, tham khảo",4.00


This step measures end-to-end response time and gives you answers to inspect. Because the LLM runs locally, generation speed will depend on your computer.

### 10. Automated Quality Grading Simulation

To automate scoring based on the rubric, we can programmatically score each QA result based on heuristic criteria:
* **Correctness**: Checks if any of the expected terms appear in the generated answer (0: None, 1: At least one, 2: Most/all expected terms).
* **Completeness**: Compares the length and detail of the answer (0: Very short or missing answer, 1: Brief or partial, 2: Elaborate response).
* **Grounding**: Compares the overlap of terms between the generated answer and retrieved source text (0: No overlap, 1: Low overlap, 2: Strong textual overlap).
* **Citation**: Detects if specific PDF source names and page numbers are formatted and present in the final output (0: None, 1: Simple reference, 2: Full source file and page citation).

In [23]:
import pandas as pd
import numpy as np

def auto_grade_qa(results_df, cases):
    graded_rows = []

    for idx, row in results_df.iterrows():
        lang = row["Language"]
        question = row["Question"]
        answer = str(row["Answer"]).lower()
        sources_str = str(row["Sources"]).lower()

        # Retrieve corresponding expected terms for this question
        expected = []
        for case in cases:
            if case["question"] == row["Question"]:
                expected = [t.lower() for t in case["expected_terms"]]
                break

        # 1. Correctness Score (Based on Expected Terms matching LLM Answer)
        matched_terms = [term for term in expected if term in answer]
        if len(expected) == 0:
            correctness = 2
        elif len(matched_terms) == 0:
            correctness = 0
        elif len(matched_terms) < len(expected):
            correctness = 1
        else:
            correctness = 2

        # 2. Completeness Score (Based on length and structures of answer text)
        word_count = len(answer.split())
        if "does not explicitly" in answer or "not find" in answer or word_count < 10:
            completeness = 0
        elif word_count < 30:
            completeness = 1
        else:
            completeness = 2

        # 3. Grounding Score (Whether the answer text relies on retrieval metadata sources)
        if len(sources_str) == 0 or "not found" in sources_str:
            grounding = 0
        elif len(matched_terms) > 0:
            grounding = 2
        else:
            grounding = 1

        # 4. Citation Score (Based on exact matching pattern for sources and pages)
        if "pdf" in answer or "trang" in answer or "page" in answer or "p." in sources_str:
            citation = 2
        elif "pdf" in sources_str:
            citation = 1
        else:
            citation = 0

        total = correctness + completeness + grounding + citation

        graded_rows.append({
            "Language": lang,
            "Question": row["Question"],
            "Correctness": correctness,
            "Completeness": completeness,
            "Grounding": grounding,
            "Citation": citation,
            "Total / 8": total
        })

    return pd.DataFrame(graded_rows)

auto_graded_df = auto_grade_qa(qa_results, test_cases)
display(auto_graded_df)

# Show average scores by language group
display(auto_graded_df.groupby("Language")[["Correctness", "Completeness", "Grounding", "Citation", "Total / 8"]].mean())

,Language,Question,Correctness,Completeness,Grounding,Citation,Total / 8
0,English,What are the prerequisites for this course?,0,2,1,2,5
1,Vietnamese,Môn học này có những điều kiện tiên quyết nào?,0,2,1,2,5
2,English,How is the final grade calculated?,0,2,1,2,5
3,Vietnamese,Môn học được đánh giá như thế nào?,0,2,1,2,5
4,English,What are the topics covered for this course?,2,2,2,2,8
5,Vietnamese,Môn học này có những nội dung gì?,1,2,2,2,7
6,English,What is the number of credits and the total ho...,2,1,2,2,7
7,Vietnamese,"Môn học này có bao nhiêu tín chỉ, tiết Lý thuy...",2,2,2,2,8
8,English,What are the required course materials,2,2,2,2,8
9,Vietnamese,Môn học này cần tài liệu gì?,1,2,2,2,7


,Correctness,Completeness,Grounding,Citation,Total / 8
Language,,,,,
English,1.2,1.8,1.6,2.0,6.6
Vietnamese,0.8,2.0,1.6,2.0,6.4


### Conclusion

This notebook demonstrates the complete RAG pipeline without an API token: PDF text extraction, chunking, multilingual embeddings, similarity-based retrieval, and LLM answer generation.

The evaluation should help answer three important questions:
1. Can the system retrieve the correct syllabus passages?

2. Can the LLM answer questions accurately without inventing information?

3. Does performance differ between English and Vietnamese?

The scores should be reported only after running the notebook on your actual PDFs. In particular, the small LLM may struggle with Vietnamese generation, and the retrieval and grounding checks are essential before treating its answers as reliable.